# Zero to Hero: EKF-SLAM with `kiss_slam`

Welcome! This notebook is a practical, runnable walkthrough of the repository's 2D EKF-SLAM pipeline.

## Table of Contents
1. [Setup & imports](#1-setup--imports)
2. [Repo tour (brief)](#2-repo-tour-brief)
3. [World + simulator](#3-world--simulator-generate-world-run-sim-visualize-landmarks)
4. [Motion model](#4-motion-model-apply-controls-visualize-trajectory)
5. [Measurement model](#5-measurement-model-generate-rangebearing-visualize-rays)
6. [EKF basics](#6-ekf-basics-tiny-2d-position-example)
7. [EKFSLAM core](#7-ekfslam-core-step-by-step)
8. [Data association](#8-data-association-when-association-is-wrong)
9. [LiveViewer](#9-liveviewer-truth-vs-estimate-with-overlays)
10. [Tuning](#10-tuning-noise-matrices-good-vs-bad)
11. [Real robot mode](#11-real-robot-mode-provider-interface-stub)

> **Runtime target:** under 2 minutes on a typical laptop.

## 1) Setup & imports

If this repository is not installed yet, run:

```bash
pip install -e .
```

Or from conda:

```bash
conda create -n kiss_slam python=3.10 -y
conda activate kiss_slam
pip install -e .
pip install notebook matplotlib numpy scipy
```

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(repo_root / 'src'))

import numpy as np
import matplotlib.pyplot as plt

from kiss_slam.types import Pose2D, ControlInput, Measurement, EKFSLAMConfig
from kiss_slam.sim.world import World2D
from kiss_slam.sim.simulator import Simulator, SimConfig
from kiss_slam.models.motion import DifferentialDriveMotionModel
from kiss_slam.models.measurement import RangeBearingMeasurementModel
from kiss_slam.ekf_slam import EKFSLAM
from kiss_slam.data_association import KnownCorrespondenceAssociator, NearestNeighborAssociator
from kiss_slam.viz.live_viewer import LiveViewer

np.random.seed(7)
plt.rcParams['figure.figsize'] = (6, 5)
print('Imports OK')

## 2) Repo tour (brief)

Core modules used below:
- `sim/world.py`: `World2D` landmark generation.
- `sim/simulator.py`: `Simulator2D`-style loop (`Simulator`) for controls, noisy odometry, and noisy measurements.
- `models/motion.py`: unicycle motion prediction + Jacobians.
- `models/measurement.py`: range/bearing prediction + Jacobians.
- `ekf_slam.py`: `EKFSLAM` filter.
- `data_association.py`: known-ID baseline + nearest-neighbor associator.
- `viz/live_viewer.py`: interactive plotting helper.

> Practical EKF-SLAM view:
> - **Predict** robot state forward with odometry + process noise.
> - **Update** with landmark range/bearing measurements + measurement noise.
> - Maintain a joint Gaussian over robot + all landmarks.

## 3) World + simulator: generate world, run sim, visualize landmarks

In [ ]:
seed = 10
world = World2D.pattern(pattern='grid', n_landmarks=16, xlim=(-8, 8), ylim=(-8, 8), seed=seed)

config = SimConfig(
    dt=0.1,
    steps=60,
    trajectory_mode='figure_eight',
    sensor_range=8.0,
    measurement_noise_std=(0.15, np.deg2rad(2.0)),
    control_noise_std=(0.03, 0.02),
)
sim = Simulator(world=world, config=config, seed=seed)

batch = sim.run()
print(f'steps: {len(batch)}, first step measurements: {len(batch[0][2])}')

In [ ]:
landmarks_xy = np.array([[lm.x, lm.y] for lm in world.landmarks])
true_xy = np.array([[step[3].x, step[3].y] for step in batch])

plt.figure()
plt.scatter(landmarks_xy[:,0], landmarks_xy[:,1], marker='x', label='landmarks')
plt.plot(true_xy[:,0], true_xy[:,1], 'k-', label='true trajectory')
plt.axis('equal'); plt.grid(True); plt.legend(); plt.title('World + trajectory')
plt.show()

**Exercise:** change `trajectory_mode` to `'rectangle'` or `'random_smooth'` and rerun.

## 4) Motion model: apply controls, visualize trajectory

In [ ]:
motion = DifferentialDriveMotionModel()
pose = np.array([0.0, 0.0, 0.0])
dt = 0.1
controls = [ControlInput(v=1.0, w=0.3*np.sin(i*0.1)) for i in range(80)]

traj = [pose.copy()]
for u in controls:
    pose = motion.predict_state(pose, u, dt)
    traj.append(pose.copy())
traj = np.array(traj)

plt.figure()
plt.plot(traj[:,0], traj[:,1], '-b')
plt.axis('equal'); plt.grid(True); plt.title('Motion model trajectory')
plt.show()

Common failure mode: wrong yaw units (degrees vs radians) causes immediate divergence.

## 5) Measurement model: generate range/bearing, visualize rays

In [ ]:
measurement_model = RangeBearingMeasurementModel()
robot = np.array([1.0, 1.0, np.deg2rad(20)])
subset = world.landmarks[:5]

plt.figure()
for lm in subset:
    lxy = np.array([lm.x, lm.y])
    z = measurement_model.predict(robot, lxy)
    plt.plot([robot[0], lxy[0]], [robot[1], lxy[1]], 'r-', alpha=0.5)
    plt.text(lxy[0], lxy[1], f'r={z[0]:.1f}m\nb={np.rad2deg(z[1]):.1f}°', fontsize=8)

plt.scatter(robot[0], robot[1], c='k', label='robot')
plt.scatter([lm.x for lm in subset], [lm.y for lm in subset], marker='x', label='landmarks')
plt.axis('equal'); plt.grid(True); plt.legend(); plt.title('Range/Bearing rays')
plt.show()

## 6) EKF basics: tiny 2D position example

Before SLAM, here is a tiny EKF-like (linear Kalman) update over a 2D position state `x=[px, py]`.
- **State covariance** expresses uncertainty.
- **Predict** increases uncertainty.
- **Update** contracts uncertainty when measurement arrives.

In [ ]:
x = np.array([0.0, 0.0])
P = np.diag([1.0, 1.0])
F = np.eye(2)
Q = np.diag([0.05, 0.05])

# Predict
x = F @ x
P = F @ P @ F.T + Q

# Measurement z = [1.2, -0.2]
z = np.array([1.2, -0.2])
H = np.eye(2)
R = np.diag([0.2, 0.2])
y = z - H @ x
S = H @ P @ H.T + R
K = P @ H.T @ np.linalg.inv(S)

x = x + K @ y
P = (np.eye(2) - K @ H) @ P

print('updated state:', x)
print('updated covariance diag:', np.diag(P))

**Exercise:** increase `R` 10x. Does the filter trust the measurement less?

## 7) EKFSLAM core: run EKFSLAM step-by-step

In [ ]:
world = World2D.random(seed=3, n_landmarks=12, xlim=(-10,10), ylim=(-10,10))
sim = Simulator(world=world, config=SimConfig(steps=50, sensor_range=9.0), seed=3)

slam = EKFSLAM(
    motion_model=DifferentialDriveMotionModel(),
    measurement_model=RangeBearingMeasurementModel(),
    assoc=KnownCorrespondenceAssociator(),
    config=EKFSLAMConfig(),
)

true_traj, est_traj = [], []
for k, (u, dt, measurements, gt_pose) in enumerate(sim.run()):
    slam.step(u, dt, measurements)
    est_pose, est_lms = slam.get_state()
    true_traj.append([gt_pose.x, gt_pose.y, gt_pose.yaw])
    est_traj.append([est_pose.x, est_pose.y, est_pose.yaw])
    if k % 10 == 0:
        print(f'step={k:02d} state_dim={slam.state.size} landmarks={sorted(est_lms.keys())[:5]}...')

true_traj = np.array(true_traj)
est_traj = np.array(est_traj)
print('final state dim:', slam.state.size)
print('final landmark count:', len(slam.landmark_states()))

In [ ]:
plt.figure()
plt.plot(true_traj[:,0], true_traj[:,1], 'k-', label='truth')
plt.plot(est_traj[:,0], est_traj[:,1], 'b--', label='estimate')
plt.scatter([lm.x for lm in world.landmarks], [lm.y for lm in world.landmarks], marker='x', c='k', label='landmarks')
plt.axis('equal'); plt.grid(True); plt.legend(); plt.title('EKF-SLAM core result')
plt.show()

Failure modes to watch:
- process noise too small -> overconfident, brittle updates,
- measurement noise too small -> aggressive, unstable corrections,
- angle wrapping mistakes -> sudden map corruption.

## 8) Data association: show what happens when association is wrong

In [ ]:
# Run two filters: correct IDs vs intentionally corrupted IDs.
world = World2D.random(seed=11, n_landmarks=10, xlim=(-8,8), ylim=(-8,8))
sim_data = Simulator(world=world, config=SimConfig(steps=40, sensor_range=8.0), seed=11).run()

slam_ok = EKFSLAM(DifferentialDriveMotionModel(), RangeBearingMeasurementModel(), assoc=KnownCorrespondenceAssociator())
slam_bad = EKFSLAM(DifferentialDriveMotionModel(), RangeBearingMeasurementModel(), assoc=KnownCorrespondenceAssociator())

truth_xy, ok_xy, bad_xy = [], [], []
for u, dt, measurements, gt in sim_data:
    slam_ok.step(u, dt, measurements)

    corrupted = []
    for m in measurements:
        wrong_id = None if m.landmark_id is None else (m.landmark_id + 1) % len(world.landmarks)
        corrupted.append(Measurement(landmark_id=wrong_id, range_m=m.range_m, bearing_rad=m.bearing_rad))
    slam_bad.step(u, dt, corrupted)

    truth_xy.append([gt.x, gt.y])
    ok_xy.append(slam_ok.robot_pose()[:2])
    bad_xy.append(slam_bad.robot_pose()[:2])

truth_xy = np.array(truth_xy); ok_xy = np.array(ok_xy); bad_xy = np.array(bad_xy)
plt.figure()
plt.plot(truth_xy[:,0], truth_xy[:,1], 'k-', label='truth')
plt.plot(ok_xy[:,0], ok_xy[:,1], 'g--', label='correct association')
plt.plot(bad_xy[:,0], bad_xy[:,1], 'r--', label='wrong association')
plt.axis('equal'); plt.grid(True); plt.legend(); plt.title('Association errors can break SLAM')
plt.show()

**Optional challenge:** swap to `NearestNeighborAssociator(gate_threshold=...)` and tune gate values.

## 9) LiveViewer: plot truth vs estimate, toggle overlays

In [ ]:
viewer = LiveViewer()
viewer.show_innovations = True
viewer.show_covariance = True

landmark_covs = {lid: slam.landmark_covariance(lid) for lid in slam.landmark_states()}
viewer.update(
    true_traj=true_traj,
    est_traj=est_traj,
    current_pose=slam.robot_pose(),
    true_landmarks=np.array([[lm.x, lm.y] for lm in world.landmarks]),
    est_landmarks=slam.landmark_states(),
    state_cov=slam.get_covariance(),
    landmark_covariances=landmark_covs,
    innovation_vectors=slam.innovation_vectors(),
)
plt.show()

Toggle fields in code:
- `viewer.show_ground_truth = False`
- `viewer.show_covariance = False`
- `viewer.show_innovations = False`

## 10) Tuning: show effect of noise matrices on stability

In [ ]:
def run_with_noise(q_scale=1.0, r_scale=1.0, seed=0):
    world = World2D.random(seed=seed, n_landmarks=14, xlim=(-10,10), ylim=(-10,10))
    sim = Simulator(world=world, config=SimConfig(steps=60, sensor_range=10.0), seed=seed)
    cfg = EKFSLAMConfig(
        process_noise=np.diag([0.05**2, 0.03**2]) * q_scale,
        measurement_noise=np.diag([0.2**2, np.deg2rad(3.0)**2]) * r_scale,
    )
    slam = EKFSLAM(DifferentialDriveMotionModel(), RangeBearingMeasurementModel(), config=cfg)

    errs = []
    for u, dt, zs, gt in sim.run():
        slam.step(u, dt, zs)
        e = slam.robot_pose()[:2] - np.array([gt.x, gt.y])
        errs.append(np.linalg.norm(e))
    return float(np.mean(errs))

rmse_good = run_with_noise(1.0, 1.0, seed=5)
rmse_bad = run_with_noise(1e-3, 1e-3, seed=5)
print(f'good settings mean position error: {rmse_good:.3f} m')
print(f'bad settings mean position error:  {rmse_bad:.3f} m')

If bad settings look unstable, that's expected: mis-specified noise is a top EKF failure mode.

## 11) Real robot mode: provider interface stub + example

In [ ]:
from dataclasses import dataclass
from typing import Protocol, Iterable

class SensorProvider(Protocol):
    def stream(self) -> Iterable[tuple[ControlInput, float, list[Measurement]]]:
        ...

@dataclass
class ReplayProvider:
    sim_outputs: list[tuple[ControlInput, float, list[Measurement], Pose2D]]

    def stream(self):
        for u, dt, measurements, _gt in self.sim_outputs:
            yield u, dt, measurements

provider = ReplayProvider(sim_outputs=sim_data)
slam_robot_mode = EKFSLAM(DifferentialDriveMotionModel(), RangeBearingMeasurementModel())
for u, dt, measurements in provider.stream():
    slam_robot_mode.step(u, dt, measurements)

print('robot-mode final pose estimate:', slam_robot_mode.robot_pose())

## Wrap-up

You now have a practical mental model:
- simulator gives controls + range/bearing landmarks,
- motion/measurement models define predict/update math,
- EKF-SLAM grows state as landmarks are discovered,
- association quality and noise tuning determine stability.

**Next challenge:** add metrics (RMSE, NIS curves) and compare Known-ID vs Nearest-Neighbor on harder trajectories.